In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v3").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Phase 5 · Domain-aware feature space

The shared-core embedding uses features applicable across organism groups. Domain-specific coordinates are fit only for the groups selected in `configs/default.yaml`. Not-applicable features are excluded rather than zero-imputed. IMSTE parameters are read from the versioned config and can be overridden explicitly below.

In [ ]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import display
from pathogen_atlas_v3.common.config import selected_organism_groups
from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.pipelines.phase5_embedding import build_phase5

CONFIG = load_yaml("configs/default.yaml")
GROUPS = selected_organism_groups(CONFIG)
MATRIX_PATH = Path(CONFIG["paths"]["release_root"]) / "species_feature_matrix.parquet"
print("Selected groups:", GROUPS)
print("Feature matrix:", MATRIX_PATH, "| exists:", MATRIX_PATH.exists())
if MATRIX_PATH.exists():
    matrix = pd.read_parquet(MATRIX_PATH)
    display(matrix.groupby("organism_group").size().rename("species rows").to_frame())
IMSTE_SETTINGS = dict(CONFIG["embedding"])
display(pd.DataFrame([IMSTE_SETTINGS]))
print("Coordinate output:", CONFIG["paths"]["embedding_root"])
print("\nImplementation called by this notebook:\n")
print(inspect.getsource(build_phase5))

## Fit the selected-domain coordinates

IMSTE runs locally. `RUN_PHASE` stays off until `.[viz]` is installed and Phase 4 has scored enough applicable features. Fitting progress is reported by the IMSTE implementation.

In [ ]:
RUN_PHASE = False
if RUN_PHASE:
    COORDINATES = build_phase5(settings_overrides=IMSTE_SETTINGS)
    display(COORDINATES.groupby("embedding_scope").size().rename("species coordinates").to_frame())
else:
    print("Set RUN_PHASE=True after inspecting the Phase 4 matrix and installing the optional viz dependencies.")